# Pipeline Tag Prediction
## Tagging the Untagged Model Cards with the Fine-Tuned RoBERTa Model

**DATASCI 266: Natural Language Processing with Deep Learning**

UC Berkeley, School of Information

---

This notebook takes the cleaned untagged model cards and runs them through the fine-tuned RoBERTa model to predict a pipeline tag and confidence score for each one, the same idea as the baseline inference notebook, just swapping in the transformer.

Loads the saved model artifacts from `Roberta_Model_Artifacts/`, no retraining:
- `config.json`
- `model.safetensors`
- `tokenizer.json`
- `tokenizer_config.json`

Steps:
1. Install/import dependencies
2. Load the fine-tuned RoBERTa model and tokenizer
3. Load the untagged dataset
4. Tokenize and predict tags with confidence scores
5. Assemble and save the final dataset
6. Quick sanity checks

## 0. Setup

In [ ]:
!pip install -q transformers accelerate

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import joblib
import warnings

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## 1. Load the Fine-Tuned RoBERTa Model and Tokenizer

Loading directly from the `Roberta_Model_Artifacts` folder on Drive. `AutoModelForSequenceClassification` and `AutoTokenizer` both read straight from that directory since it has `config.json`, `model.safetensors`, `tokenizer.json`, and `tokenizer_config.json`.

`training_args.bin` isn't needed here, that's leftover from the `Trainer` checkpoint and has no bearing on inference.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/266-pipeline-tag-prediction'
ROBERTA_DIR = f'{DRIVE_DIR}/roberta_pipeline_tag_model'

tokenizer = AutoTokenizer.from_pretrained(ROBERTA_DIR)
model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_DIR)
model.to(device)
model.eval()

print('Loaded RoBERTa model and tokenizer from', ROBERTA_DIR)
print(f'Number of labels: {model.config.num_labels}')

Mounted at /content/drive


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded RoBERTa model and tokenizer from /content/drive/MyDrive/266-pipeline-tag-prediction/roberta_pipeline_tag_model
Number of labels: 10


## 2. Resolve Label Mapping

Checking whether `config.json` already has real tag names in `id2label`, or just generic placeholders like `LABEL_0`. If it's generic, we fall back to `label_encoder.joblib` from the baseline artifacts, since both models were trained on the same label encoding.

In [ ]:
id2label = model.config.id2label
print('id2label from model config:')
print(id2label)

generic_labels = all(str(v).startswith('LABEL_') for v in id2label.values())

if generic_labels:
    print()
    print('Config only has generic labels, falling back to label_encoder.joblib')
    label_encoder = joblib.load(f'{DRIVE_DIR}/label_encoder.joblib')
    # label_encoder.classes_ is ordered by the same integer index used during training
    id2label = {i: cls for i, cls in enumerate(label_encoder.classes_)}
    print('Resolved id2label from label encoder:')
    print(id2label)
else:
    print()
    print('Config already has real tag names, using as is.')

id2label from model config:
{0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2', 3: 'LABEL_3', 4: 'LABEL_4', 5: 'LABEL_5', 6: 'LABEL_6', 7: 'LABEL_7', 8: 'LABEL_8', 9: 'LABEL_9'}

Config only has generic labels, falling back to label_encoder.joblib
Resolved id2label from label encoder:
{0: 'automatic-speech-recognition', 1: 'image-classification', 2: 'image-text-to-text', 3: 'robotics', 4: 'sentence-similarity', 5: 'text-classification', 6: 'text-generation', 7: 'text-to-image', 8: 'token-classification', 9: 'translation'}


## 3. Load the Untagged Dataset

Same cleaned untagged dataset used for the baseline predictions. Columns are `modelId`, `text`, `char_len`, `word_count`, model input is `text`.

In [ ]:
df_untagged = pd.read_parquet(f'{DRIVE_DIR}/model_cards_untagged_cleaned.parquet')

print(f'Shape: {df_untagged.shape}')
df_untagged.head()

Shape: (115620, 4)


,modelId,text,char_len,word_count
0,DreadPoor/Kitsch_Late_ALT-TEST-Q5_K_M-GGUF,# DreadPoor/Kitsch_Late_ALT-TEST-Q5_K_M-GGUF\n...,1820,182
1,namlevan888/blockassist-bc-lethal_durable_rave...,# Gensyn BlockAssist\n\nGensyn's BlockAssist i...,169,17
2,ElenaSenger/career-path-representation-mpnet-d...,# career-path-representation-mpnet-decorte\nTh...,842,67
3,priorcomputers/llama-3.2-1b-instruct-cn-dat-kr...,# llama-3.2-1b-instruct-cn-dat-kr0.05-a1.0-cre...,1436,139
4,mradermacher/ACC-Qwen3-30B-A3B-i1-GGUF,## About\n\n<!-- ### quantize_version: 2 -->\n...,5703,488


## 4. Tokenize and Predict

Running inference in batches to avoid blowing up GPU memory on the full untagged set. RoBERTa's `512`-token limit means longer cards get truncated, same tradeoff as during training. For each card we take the softmax over the 10 classes, the predicted tag is the argmax, and the confidence is that max probability.

In [ ]:
BATCH_SIZE = 32
MAX_LENGTH = 512

texts = df_untagged['text'].tolist()

all_pred_idx = []
all_confidence = []

with torch.no_grad():
    for i in range(0, len(texts), BATCH_SIZE):
        batch_texts = texts[i:i + BATCH_SIZE]

        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors='pt'
        ).to(device)

        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)

        batch_pred_idx = torch.argmax(probs, dim=-1).cpu().numpy()
        batch_confidence = torch.max(probs, dim=-1).values.cpu().numpy()

        all_pred_idx.extend(batch_pred_idx.tolist())
        all_confidence.extend(batch_confidence.tolist())

        if (i // BATCH_SIZE) % 20 == 0:
            print(f'Processed {i + len(batch_texts)} / {len(texts)}')

print('Inference complete.')

Processed 32 / 115620
Processed 672 / 115620
Processed 1312 / 115620
Processed 1952 / 115620
Processed 2592 / 115620
Processed 3232 / 115620
Processed 3872 / 115620
Processed 4512 / 115620
Processed 5152 / 115620
Processed 5792 / 115620
Processed 6432 / 115620
Processed 7072 / 115620
Processed 7712 / 115620
Processed 8352 / 115620
Processed 8992 / 115620
Processed 9632 / 115620
Processed 10272 / 115620
Processed 10912 / 115620
Processed 11552 / 115620
Processed 12192 / 115620
Processed 12832 / 115620
Processed 13472 / 115620
Processed 14112 / 115620
Processed 14752 / 115620
Processed 15392 / 115620
Processed 16032 / 115620
Processed 16672 / 115620
Processed 17312 / 115620
Processed 17952 / 115620
Processed 18592 / 115620
Processed 19232 / 115620
Processed 19872 / 115620
Processed 20512 / 115620
Processed 21152 / 115620
Processed 21792 / 115620
Processed 22432 / 115620
Processed 23072 / 115620
Processed 23712 / 115620
Processed 24352 / 115620
Processed 24992 / 115620
Processed 25632 / 1

In [ ]:
df_untagged['predicted_tag'] = [id2label[idx] for idx in all_pred_idx]
df_untagged['confidence'] = all_confidence
df_untagged['confidence_pct'] = (df_untagged['confidence'] * 100).round(2)

print('Predictions complete.')
df_untagged[['modelId', 'predicted_tag', 'confidence_pct']].head(10)

Predictions complete.


,modelId,predicted_tag,confidence_pct
0,DreadPoor/Kitsch_Late_ALT-TEST-Q5_K_M-GGUF,text-generation,91.63
1,namlevan888/blockassist-bc-lethal_durable_rave...,text-generation,94.51
2,ElenaSenger/career-path-representation-mpnet-d...,sentence-similarity,98.98
3,priorcomputers/llama-3.2-1b-instruct-cn-dat-kr...,text-generation,99.61
4,mradermacher/ACC-Qwen3-30B-A3B-i1-GGUF,text-generation,96.49
5,4everStudent/Qwen3-4B-GRPO-chess-puzzle,text-generation,99.73
6,javasop/orbital-cli,text-generation,83.80
7,little1d/C,token-classification,91.88
8,munish0838/Qwen-2.5-1.5B-cenv-trl-grpo-v3,text-generation,90.89
9,phanerozoic/threshold-parity6,text-classification,88.53


A quick look at the confidence distribution and predicted tag counts, useful to sanity check whether the model is confident overall or hedging a lot on this unseen population of cards.

In [ ]:
print(df_untagged['confidence_pct'].describe())
print()
print(df_untagged['predicted_tag'].value_counts())

count    115620.000000
mean         89.402844
std          15.849872
min          19.260000
25%          85.460000
50%          97.860000
75%          99.580000
max          99.910000
Name: confidence_pct, dtype: float64

predicted_tag
text-generation                 76206
image-text-to-text              10899
text-classification              6630
text-to-image                    5657
image-classification             4878
automatic-speech-recognition     4451
robotics                         2684
token-classification             2092
sentence-similarity              1299
translation                       824
Name: count, dtype: int64


In [ ]:
# Spot check a handful of low-confidence predictions, these are the ones most worth a manual look later
df_untagged.sort_values('confidence_pct').head(5)[['modelId', 'predicted_tag', 'confidence_pct', 'text']]

,modelId,predicted_tag,confidence_pct,text
67678,braindecode/STEEGFormer-large,sentence-similarity,19.26,# STEEGFormer (large)\n\nViT-MAE EEG foundatio...
3727,taiypeo/bart-large-gigaword-rouge-3-loss-diffe...,token-classification,19.64,<!-- This model card has been generated automa...
96820,ztiganj/verl-edits-counterfact_cot_correctness...,text-generation,19.81,# ztiganj/verl-edits-counterfact_cot_correctne...
34683,playtranslate/ocr-models,token-classification,19.86,# PlayTranslate OCR models\n\nOn-device OCR mo...
80599,Comfy-Org/Real-ESRGAN_repackaged,text-generation,19.89,# Real-ESRGAN Repackaged\nThis repository cont...


## 5. Assemble and Save the Final Dataset

Final columns: `modelId`, the card text, and the predicted tag with confidence. Keeping `predicted_tag` and `confidence_pct` separate rather than folding them into one string, so confidence stays usable as a number for filtering or thresholding later, same convention as the baseline output.

Saved as both a parquet and a CSV, matching the baseline inference notebook.

In [ ]:
final_df = df_untagged[['modelId', 'text', 'predicted_tag', 'confidence_pct']].copy()
final_df = final_df.rename(columns={'text': 'card_text_clean'})

print(f'Final dataset shape: {final_df.shape}')
final_df.head()

Final dataset shape: (115620, 4)


,modelId,card_text_clean,predicted_tag,confidence_pct
0,DreadPoor/Kitsch_Late_ALT-TEST-Q5_K_M-GGUF,# DreadPoor/Kitsch_Late_ALT-TEST-Q5_K_M-GGUF\n...,text-generation,91.63
1,namlevan888/blockassist-bc-lethal_durable_rave...,# Gensyn BlockAssist\n\nGensyn's BlockAssist i...,text-generation,94.51
2,ElenaSenger/career-path-representation-mpnet-d...,# career-path-representation-mpnet-decorte\nTh...,sentence-similarity,98.98
3,priorcomputers/llama-3.2-1b-instruct-cn-dat-kr...,# llama-3.2-1b-instruct-cn-dat-kr0.05-a1.0-cre...,text-generation,99.61
4,mradermacher/ACC-Qwen3-30B-A3B-i1-GGUF,## About\n\n<!-- ### quantize_version: 2 -->\n...,text-generation,96.49


In [ ]:
final_df.to_parquet(f'{DRIVE_DIR}/untagged_model_cards_predicted_roberta.parquet', index=False)
final_df.to_csv(f'{DRIVE_DIR}/untagged_model_cards_predicted_roberta.csv', index=False)

print(f'Saved to {DRIVE_DIR}/untagged_model_cards_predicted_roberta.parquet')
print(f'Saved to {DRIVE_DIR}/untagged_model_cards_predicted_roberta.csv')

Saved to /content/drive/MyDrive/266-pipeline-tag-prediction/untagged_model_cards_predicted_roberta.parquet
Saved to /content/drive/MyDrive/266-pipeline-tag-prediction/untagged_model_cards_predicted_roberta.csv


## 6. Quick Sanity Checks

Comparing RoBERTa's predicted tag distribution against the baseline's, if you've already run the baseline inference notebook and have `untagged_model_cards_predicted.parquet` on Drive. Big divergences between the two are worth a closer look, since they're predicting on the exact same untagged cards.

In [ ]:
import os

baseline_path = f'{DRIVE_DIR}/untagged_model_cards_predicted.parquet'

if os.path.exists(baseline_path):
    baseline_df = pd.read_parquet(baseline_path)
    merged = final_df.merge(
        baseline_df[['modelId', 'predicted_tag', 'confidence_pct']],
        on='modelId',
        suffixes=('_roberta', '_baseline')
    )
    agreement = (merged['predicted_tag_roberta'] == merged['predicted_tag_baseline']).mean()
    print(f'Agreement rate between RoBERTa and baseline predictions: {agreement:.2%}')

    disagreements = merged[merged['predicted_tag_roberta'] != merged['predicted_tag_baseline']]
    print(f'Number of disagreements: {len(disagreements)}')
    disagreements[['modelId', 'predicted_tag_baseline', 'confidence_pct_baseline',
                    'predicted_tag_roberta', 'confidence_pct_roberta']].head(10)
else:
    print('Baseline predictions file not found on Drive, skipping comparison.')
    print('Run NH_Untagged_Inference.ipynb first if you want this comparison.')

Agreement rate between RoBERTa and baseline predictions: 79.36%
Number of disagreements: 23866


### Notes for later use

- These are RoBERTa-model predictions, not ground truth. Treat `confidence_pct` as a filter, not a guarantee, low-confidence rows are the ones most likely to be genuinely ambiguous or out of distribution relative to the training set.
- RoBERTa truncates at 512 tokens, so longer model cards lose their tail end during prediction. If a card is long and its prediction looks off, that truncation is worth checking first.
- Since the model was trained only on the top 10 pipeline tags, every prediction here is forced into one of those 10 categories, even if a card actually belongs to a tag outside that set.
- If `Roberta_Model_Artifacts/` is missing `config.json`, `model.safetensors`, `tokenizer.json`, or `tokenizer_config.json`, this notebook will fail at Section 1. Re-save the fine-tuned model with `model.save_pretrained()` and `tokenizer.save_pretrained()` if that happens.